# Coffee17 Master Screening — Analysis Package Only

Use this notebook **only after the master screening has already finished** and its Kaggle saved output is attached as an Input.

This notebook does **not train**, does **not require GPU**, and never loads model checkpoints. It only extracts compact result artifacts into one ZIP for external analysis.


In [ ]:
NOTEBOOK_BUILD = "MASTER-SCREENING-ANALYSIS-ONLY-V2"
import json, shutil, zipfile
from pathlib import Path

INPUT = Path("/kaggle/input")
WORK = Path("/kaggle/working")
OUTPUT_ZIP = WORK / "coffee17-master-screening-analysis-package.zip"
STAGE = WORK / "_coffee17_master_analysis_only_stage"

print("NOTEBOOK_BUILD:", NOTEBOOK_BUILD)
print("TRAINING: DISABLED")
print("GPU: NOT REQUIRED")

shutil.rmtree(STAGE, ignore_errors=True)
STAGE.mkdir(parents=True, exist_ok=True)
if OUTPUT_ZIP.exists():
    OUTPUT_ZIP.unlink()

# 1) Prefer an already-created compact package from the attached saved output.
existing_packages = sorted(
    p for p in INPUT.rglob("coffee17-master-screening-analysis-package.zip")
    if p.is_file()
)
if len(existing_packages) > 1:
    raise RuntimeError(
        "Lebih dari satu compact analysis package ditemukan: "
        + str([str(p) for p in existing_packages])
    )

if existing_packages:
    source = existing_packages[0]
    shutil.copy2(source, OUTPUT_ZIP)
    print("REUSED EXISTING ANALYSIS PACKAGE:", source)
else:
    # 2) Otherwise reconstruct a compact package from the completed Saved Version.
    # Kaggle may flatten / rename the original /kaggle/working project directory
    # when exposing notebook outputs as an Input, so do NOT depend on the original
    # folder name. Locate the unique final master summary first, then infer root.
    summaries = sorted(
        p for p in INPUT.rglob("master_screening_v1.json")
        if p.is_file()
    )
    if not summaries:
        # Show useful top-level diagnostics instead of silently retraining.
        top_level = sorted(str(p) for p in INPUT.iterdir())
        raise RuntimeError(
            "master_screening_v1.json tidak ditemukan di Saved Version yang di-attach. "
            "Notebook ini TIDAK akan training ulang. Top-level /kaggle/input: "
            + str(top_level)
        )
    if len(summaries) != 1:
        raise RuntimeError(
            "Lebih dari satu master_screening_v1.json ditemukan: "
            + str([str(p) for p in summaries])
        )

    summary = summaries[0]
    if summary.parent.name == "analysis":
        project = summary.parent.parent
    else:
        # Defensive fallback for a flattened Saved Version layout.
        project = summary.parent

    print("FOUND MASTER SUMMARY:", summary)
    print("INFERRED SAVED OUTPUT ROOT:", project)

    # Validate that all five fold results exist before packaging. If Kaggle
    # flattened the project tree, locate each fold_result.json under the inferred root.
    fold_results = []
    for fold in range(1, 6):
        expected = project / "experiments" / f"fold_{fold}" / "seed42" / "fold_result.json"
        if expected.is_file():
            path = expected
        else:
            matches = sorted(
                p for p in project.rglob("fold_result.json")
                if p.parent.name == "seed42" and p.parent.parent.name == f"fold_{fold}"
            )
            if len(matches) != 1:
                raise RuntimeError(
                    f"Fold {fold} result tidak unik/ditemukan: "
                    + str([str(p) for p in matches])
                )
            path = matches[0]
        fold_results.append(path)

    def copy_file(src: Path, rel: Path):
        dst = STAGE / rel
        dst.parent.mkdir(parents=True, exist_ok=True)
        shutil.copy2(src, dst)

    copy_file(summary, Path("analysis/master_screening_v1.json"))

    # Compact per-fold/per-arm evidence only; NEVER copy checkpoints.
    for fold, fold_result in enumerate(fold_results, start=1):
        fold_root = fold_result.parent
        dst_fold = Path("experiments") / f"fold_{fold}" / "seed42"
        copy_file(fold_result, dst_fold / "fold_result.json")

        for arm in sorted(p for p in fold_root.iterdir() if p.is_dir()):
            dst_arm = dst_fold / arm.name
            for name in ("run_contract.json", "run_config.yaml", "history.json"):
                src = arm / name
                if src.is_file():
                    copy_file(src, dst_arm / name)

            validation = arm / "validation"
            if validation.is_dir():
                for src in sorted(p for p in validation.rglob("*") if p.is_file()):
                    copy_file(src, dst_arm / "validation" / src.relative_to(validation))

    # Explicitly record what is and is not included.
    files = [
        {
            "path": str(p.relative_to(STAGE)),
            "bytes": p.stat().st_size,
        }
        for p in sorted(q for q in STAGE.rglob("*") if q.is_file())
    ]
    (STAGE / "PACKAGE_MANIFEST.json").write_text(
        json.dumps(
            {
                "format": "coffee17.master_screening.analysis_only_package.v1",
                "training_performed": False,
                "checkpoints_included": False,
                "source_project": str(project),
                "files": files,
            },
            indent=2,
        ) + "\n",
        encoding="utf-8",
    )

    shutil.make_archive(
        str(OUTPUT_ZIP.with_suffix("")),
        "zip",
        root_dir=STAGE,
    )
    print("BUILT ANALYSIS PACKAGE FROM:", project)

if not OUTPUT_ZIP.is_file():
    raise RuntimeError("Analysis ZIP gagal dibuat.")

with zipfile.ZipFile(OUTPUT_ZIP) as zf:
    names = zf.namelist()
    checkpoint_names = [n for n in names if n.endswith(".pt")]
    if checkpoint_names:
        raise RuntimeError(
            "Compact package tidak boleh mengandung checkpoint: "
            + str(checkpoint_names[:10])
        )
    if not any(n.endswith("master_screening_v1.json") for n in names):
        raise RuntimeError("Master summary tidak ditemukan di analysis package.")

print("READY:", OUTPUT_ZIP)
print("SIZE_MB:", round(OUTPUT_ZIP.stat().st_size / 1024 / 1024, 2))
print("FILES:", len(names))
print("CHECKPOINTS_INCLUDED: 0")
print("UPLOAD THIS ZIP TO CHAT FOR ANALYSIS.")
